In [ ]:
import os


!unzip -q /content/SAMWISE_code.zip -d /content/
%cd /content/SAMWISE

# 2. Install core CV and NLP libraries
!pip install -q opencv-python pycocotools moviepy tqdm einops spacy py3_wget scikit-learn sacrebleu bitarray gdown git+https://github.com/facebookresearch/segment-anything-2.git

# 3. Install HuggingFace libraries for Qwen Edge-LLM
!pip install -q transformers accelerate bitsandbytes

# 4. Setup Spacy
!python -m spacy download en_core_web_sm

# 5. Download the pre-trained weights
!mkdir -p pretrain
!gdown 1Molt2up2bP41ekeczXWQU-LWTskKJOV2 -O pretrain/

print("Environment initialized and weights downloaded.")

/content/SAMWISE
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.6/343.6 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 115.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtim

In [ ]:

import os
import re

print("Starting Fairseq sterilization...")

# 1. Amputate Eager Loading & Hydra Init
init_files = {
    "fairseq/__init__.py": ("hydra_init()", "# hydra_init()"),
    "fairseq/models/__init__.py": ('import_models(models_dir, "fairseq.models")', '# import_models(models_dir, "fairseq.models")'),
    "fairseq/criterions/__init__.py": ('importlib.import_module("fairseq.criterions." + file_name)', '# importlib.import_module("fairseq.criterions." + file_name)')
}

for filepath, (target, replacement) in init_files.items():
    if os.path.exists(filepath):
        with open(filepath, "r") as f:
            content = f.read()
        with open(filepath, "w") as f:
            f.write(content.replace(target, replacement))

# 2. Nuclear Dataclass Patcher (Mutable Defaults)
def patch_dataclasses(directory):
    count = 0
    for root, _, files in os.walk(directory):
        for file in files:
            if file.endswith(".py"):
                filepath = os.path.join(root, file)
                with open(filepath, "r", encoding="utf-8") as f:
                    content = f.read()

                if "@dataclass" in content:
                    if "from dataclasses import dataclass" in content and "field" not in content:
                        content = content.replace("from dataclasses import dataclass", "from dataclasses import dataclass, field")

                    patched_content = re.sub(
                        r'([a-zA-Z0-9_]+):\s*([a-zA-Z0-9_]+)\s*=\s*\2\([^)]*\)',
                        r'\1: \2 = field(default_factory=\2)',
                        content
                    )

                    if patched_content != content:
                        with open(filepath, "w", encoding="utf-8") as f:
                            f.write(patched_content)
                        count += 1
    return count

fixed_files = patch_dataclasses("fairseq")
print(f"✅ Fairseq Sterilization Complete. Fixed {fixed_files} files.")

Starting Fairseq sterilization...
✅ Fairseq Sterilization Complete. Fixed 0 files.


In [3]:
import os
import re

# Ensure we are in the right directory
%cd /content/SAMWISE

print("Applying surgical Fairseq Python 3.12 patches...")

# 1. Fix configs.py (The current crash)
with open("fairseq/dataclass/configs.py", "r") as f: c = f.read()
if "from dataclasses import field" not in c:
    c = c.replace("from dataclasses import dataclass", "from dataclasses import dataclass, field")
c = re.sub(r'([a-zA-Z0-9_]+):\s*([a-zA-Z0-9_]+Config)\s*=\s*\2\(\)', r'\1: \2 = field(default_factory=\2)', c)
with open("fairseq/dataclass/configs.py", "w") as f: f.write(c)

# 2. Bypass Hydra
with open("fairseq/__init__.py", "r") as f: c = f.read()
c = c.replace("hydra_init()", "# hydra_init()")
with open("fairseq/__init__.py", "w") as f: f.write(c)

# 3. Amputate Eager Loading (Models & Criterions)
with open("fairseq/models/__init__.py", "r") as f: c = f.read()
c = c.replace('import_models(models_dir, "fairseq.models")', '# import_models(models_dir, "fairseq.models")')
with open("fairseq/models/__init__.py", "w") as f: f.write(c)

with open("fairseq/criterions/__init__.py", "r") as f: c = f.read()
c = c.replace('importlib.import_module("fairseq.criterions." + file_name)', '# importlib.import_module("fairseq.criterions." + file_name)')
with open("fairseq/criterions/__init__.py", "w") as f: f.write(c)

# 4. Fix transformer_config.py
with open("fairseq/models/transformer/transformer_config.py", "r") as f: c = f.read()
if "from dataclasses import field" not in c:
    c = c.replace("from dataclasses import dataclass", "from dataclasses import dataclass, field")
c = re.sub(r'([a-zA-Z0-9_]+):\s*([a-zA-Z0-9_]+Config)\s*=\s*\2\(\)', r'\1: \2 = field(default_factory=\2)', c)
with open("fairseq/models/transformer/transformer_config.py", "w") as f: f.write(c)

print("✅ All hardcoded patches applied securely!")

/content/SAMWISE
Applying surgical Fairseq Python 3.12 patches...
✅ All hardcoded patches applied securely!


In [5]:
import os

%cd /content/SAMWISE

def brute_force_fix(filepath):
    if not os.path.exists(filepath): return
    with open(filepath, "r") as f: lines = f.readlines()
    with open(filepath, "w") as f:
        for line in lines:
            # If the line assigns a Config object, override it line-by-line
            if "Config" in line and "=" in line and "field(" not in line and "@" not in line and ":" in line:
                try:
                    indent = line[:len(line) - len(line.lstrip())]
                    var_name = line.split(":")[0].strip()
                    type_name = line.split(":")[1].split("=")[0].strip()
                    if "Config" in type_name:
                        f.write(f"{indent}{var_name}: {type_name} = field(default_factory={type_name})\n")
                        continue
                except:
                    pass
            f.write(line)

brute_force_fix("fairseq/models/transformer/transformer_config.py")
brute_force_fix("fairseq/dataclass/configs.py")
print("✅ Brute-force Python 3.12 patch applied!")

/content/SAMWISE
✅ Brute-force Python 3.12 patch applied!


In [9]:
import re
import os

%cd /content/SAMWISE

filepath = "fairseq/models/transformer/transformer_config.py"

with open(filepath, "r") as f:
    content = f.read()

# 1. Ensure 'field' is imported
if "from dataclasses import field" not in content:
    content = content.replace("from dataclasses import dataclass", "from dataclasses import dataclass, field")

# 2. Structure-Aware Block Replacement using Lookaheads
# This deletes EVERYTHING after the equals sign until it sees the next indented variable or decorator.
content = re.sub(
    r'(quant_noise:\s*QuantNoiseConfig\s*=).*?(?=\n\s+[a-zA-Z_]+:|\n\s*@)',
    r'\1 field(default_factory=QuantNoiseConfig)',
    content,
    flags=re.DOTALL
)

content = re.sub(
    r'(encoder:\s*EncDecBaseConfig\s*=).*?(?=\n\s+[a-zA-Z_]+:|\n\s*@)',
    r'\1 field(default_factory=EncDecBaseConfig)',
    content,
    flags=re.DOTALL
)

content = re.sub(
    r'(decoder:\s*EncDecBaseConfig\s*=).*?(?=\n\s+[a-zA-Z_]+:|\n\s*@)',
    r'\1 field(default_factory=EncDecBaseConfig)',
    content,
    flags=re.DOTALL
)

with open(filepath, "w") as f:
    f.write(content)

print("✅ Structure-Aware Patch applied! Nested configs have been eradicated.")

/content/SAMWISE
✅ Structure-Aware Patch applied! Nested configs have been eradicated.


In [11]:
%%writefile /content/SAMWISE/mlx_lm.py
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def load(model_path, **kwargs):
    """Intercepts MLX load and routes to HuggingFace on CUDA"""
    print(f"🔄 Intercepted MLX load request for '{model_path}'.")
    print("🚀 Routing to HuggingFace Transformers on CUDA...")

    # Force the standard Hugging Face model ID for Qwen2.5 1.5B
    hf_model_id = "Qwen/Qwen2.5-1.5B-Instruct"

    # Replicate 4-bit quantization for the Nvidia T4 GPU
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16
    )

    tokenizer = AutoTokenizer.from_pretrained(hf_model_id)
    model = AutoModelForCausalLM.from_pretrained(
        hf_model_id,
        quantization_config=quantization_config,
        device_map="auto"
    )
    return model, tokenizer

def generate(model, tokenizer, prompt, max_tokens=200, **kwargs):
    """Intercepts MLX generate and executes via standard transformers"""
    # Ensure inputs are sent to the CUDA GPU
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            pad_token_id=tokenizer.eos_token_id
        )

    # Slice the output to return only the generated explanation, not the original prompt
    input_length = inputs.input_ids.shape[1]
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    return response.strip()

Writing /content/SAMWISE/mlx_lm.py


In [12]:
%cd /content/SAMWISE

!python inference_demo.py \
    --device "cuda" \
    --input_path "../data/processed_clip.mp4" \
    --text_prompts "बिल्ली पेड़ पर चढ़ रही है"

/content/SAMWISE
2026-05-28 19:06:08.272756: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading explainability model (mlx-community/Qwen2.5-1.5B-Instruct-4bit) into unified memory...
🔄 Intercepted MLX load request for 'mlx-community/Qwen2.5-1.5B-Instruct-4bit'.
🚀 Routing to HuggingFace Transformers on CUDA...
config.json: 100% 660/660 [00:00<00:00, 3.50MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 23.6MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 64.7MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 119MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 150MB/s]
model.safetensors: 100% 3.09G/3.09G [00:38<00:00, 80.2MB/s]
Loading weights: 100% 338/338 [00:06<00:00, 54.90it/s, Materializing param=model.norm.weight] 
generation

In [13]:
import os

%cd /content/SAMWISE

# 1. Upgrade OmegaConf from master to inherently fix the Python 3.12 _MISSING_TYPE bug
!pip install -q git+https://github.com/omry/omegaconf.git

# 2. Turn Hydra back on
with open("fairseq/__init__.py", "r") as f: c = f.read()
c = c.replace("# hydra_init()", "hydra_init()")
with open("fairseq/__init__.py", "w") as f: f.write(c)

# 3. Explicitly route the Cross Entropy criterion (bypassing the broken loop)
with open("fairseq/criterions/__init__.py", "r") as f: c = f.read()
if "import fairseq.criterions.cross_entropy" not in c:
    c += "\nimport fairseq.criterions.cross_entropy\n"
with open("fairseq/criterions/__init__.py", "w") as f: f.write(c)

# 4. Explicitly route RoBERTa (bypassing the broken loop)
with open("fairseq/models/__init__.py", "r") as f: c = f.read()
if "import fairseq.models.roberta" not in c:
    c += "\nimport fairseq.models.roberta\n"
with open("fairseq/models/__init__.py", "w") as f: f.write(c)

print("✅ Configuration dependencies restored securely! OmegaConf upgraded.")

/content/SAMWISE
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hydra-core 1.3.2 requires omegaconf<2.4,>=2.2, but you have omegaconf 2.4.0.dev11 which is incompatible.
✅ Configuration dependencies restored securely! OmegaConf upgraded.


In [15]:
import sys
import os

%cd /content/SAMWISE

# 1. Sandbox initialize.py to silently ignore Python 3.12 _MISSING_TYPE configs
filepath = "fairseq/dataclass/initialize.py"
with open(filepath, "r") as f:
    content = f.read()

original_store = "cs.store(name=k, node=v)"
safe_store = """try:
            cs.store(name=k, node=v)
        except Exception:
            pass"""

if "except Exception:" not in content and original_store in content:
    content = content.replace(original_store, safe_store)
    with open(filepath, "w") as f:
        f.write(content)
    print("✅ Hydra Sandboxed: Corrupted configs will now be ignored.")

# 2. Flush the old, broken modules from Colab's active RAM
flushed = 0
for m in list(sys.modules.keys()):
    if m.startswith("fairseq") or m.startswith("omegaconf") or m.startswith("hydra"):
        del sys.modules[m]
        flushed += 1

print(f"✅ Flushed {flushed} cached modules from memory. Environment is completely clean.")

/content/SAMWISE
✅ Hydra Sandboxed: Corrupted configs will now be ignored.
✅ Flushed 0 cached modules from memory. Environment is completely clean.


In [17]:
import os
import re

%cd /content/SAMWISE

filepath = "fairseq/dataclass/initialize.py"
with open(filepath, "r") as f:
    content = f.read()

# 1. Strip out the broken, misaligned try/except block
content = re.sub(
    r'[ \t]*try:\n[ \t]*cs\.store\(name=k, node=v\)\n[ \t]*except Exception:\n[ \t]*pass',
    '    cs.store(name=k, node=v)',
    content
)

# 2. Re-apply it using dynamic, spacing-aware indentation
def apply_safe_store(match):
    indent = match.group(1) # Grab the exact leading spaces
    return f"{indent}try:\n{indent}    cs.store(name=k, node=v)\n{indent}except Exception:\n{indent}    pass"

content = re.sub(r'^([ \t]*)cs\.store\(name=k, node=v\)', apply_safe_store, content, flags=re.MULTILINE)

with open(filepath, "w") as f:
    f.write(content)

print("✅ Indentation fixed! The Hydra sandbox is now perfectly aligned.")

/content/SAMWISE
✅ Indentation fixed! The Hydra sandbox is now perfectly aligned.


In [18]:
%cd /content/SAMWISE

!python inference_demo.py \
    --device "cuda" \
    --input_path "../data/processed_clip.mp4" \
    --text_prompts "बिल्ली पेड़ पर चढ़ रही है"

/content/SAMWISE
2026-05-28 19:16:38.533005: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Traceback (most recent call last):
  File "/content/SAMWISE/inference_demo.py", line 22, in <module>
    from models.samwise import build_samwise
  File "/content/SAMWISE/models/samwise.py", line 13, in <module>
    from fairseq.models.roberta import RobertaModel
  File "/content/SAMWISE/fairseq/__init__.py", line 29, in <module>
    from fairseq.dataclass.initialize import hydra_init
  File "/content/SAMWISE/fairseq/dataclass/initialize.py", line 24
    try:
IndentationError: expected an indented block after 'try' statement on line 23


In [21]:
import os

%cd /content/SAMWISE

# 1. Extract a fresh, uncorrupted version of initialize.py from your zip file
!unzip -o -q /content/SAMWISE_code.zip "SAMWISE/fairseq/dataclass/initialize.py" -d /content/

# 2. Apply a flawless line-by-line patch (Zero Regex)
filepath = "fairseq/dataclass/initialize.py"
with open(filepath, "r") as f:
    lines = f.readlines()

with open(filepath, "w") as f:
    for line in lines:
        # When we find the target line, replace it with the sandboxed version
        if "cs.store(name=k, node=v)" in line:
            # Dynamically grab the exact whitespace indentation of the original line
            indent = line[:len(line) - len(line.lstrip())]
            f.write(f"{indent}try:\n")
            f.write(f"{indent}    cs.store(name=k, node=v)\n")
            f.write(f"{indent}except Exception:\n")
            f.write(f"{indent}    pass\n")
        else:
            f.write(line)

print("✅ File restored and sandboxed with mathematically perfect indentation.")

# 3. Explicitly route the Masked LM criterion (to fix KeyError: 'masked_lm')
filepath_criterions = "fairseq/criterions/__init__.py"
with open(filepath_criterions, "r") as f:
    c = f.read()
if "import fairseq.criterions.masked_lm" not in c:
    c += "\nimport fairseq.criterions.masked_lm\n"
with open(filepath_criterions, "w") as f:
    f.write(c)
print("✅ Masked LM criterion explicitly imported.")

/content/SAMWISE
✅ File restored and sandboxed with mathematically perfect indentation.
✅ Masked LM criterion explicitly imported.


In [23]:
import os

%cd /content/SAMWISE

# Explicitly route the Masked LM criterion (bypassing the broken loop)
filepath = "fairseq/criterions/__init__.py"
with open(filepath, "r") as f:
    c = f.read()

if "import fairseq.criterions.masked_lm" not in c:
    c += "\nimport fairseq.criterions.masked_lm\n"

with open(filepath, "w") as f:
    f.write(c)

print("✅ Masked LM criterion routed successfully!")

/content/SAMWISE
✅ Masked LM criterion routed successfully!


In [24]:
%cd /content/SAMWISE

!python inference_demo.py \
    --device "cuda" \
    --input_path "../data/processed_clip.mp4" \
    --text_prompts "बिल्ली पेड़ पर चढ़ रही है"

/content/SAMWISE
2026-05-28 19:20:52.595175: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Traceback (most recent call last):
  File "/content/SAMWISE/inference_demo.py", line 22, in <module>
    from models.samwise import build_samwise
  File "/content/SAMWISE/models/samwise.py", line 13, in <module>
    from fairseq.models.roberta import RobertaModel
  File "/content/SAMWISE/fairseq/__init__.py", line 33, in <module>
    import fairseq.criterions  # noqa
    ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/SAMWISE/fairseq/criterions/__init__.py", line 40, in <module>
    import fairseq.criterions.masked_lm
  File "/content/SAMWISE/fairseq/criterions/masked_lm.py", line 11, in <module>
    from fairseq import modules, utils
  File "/content/SAMWISE/

In [25]:
import os

%cd /content/SAMWISE

# 1. Remove the toxic circular injections from fairseq/__init__.py files
models_init = "fairseq/models/__init__.py"
with open(models_init, "r") as f: c = f.read()
c = c.replace("import fairseq.models.roberta", "")
with open(models_init, "w") as f: f.write(c)

criterions_init = "fairseq/criterions/__init__.py"
with open(criterions_init, "r") as f: c = f.read()
c = c.replace("import fairseq.criterions.masked_lm", "")
c = c.replace("import fairseq.criterions.cross_entropy", "")
with open(criterions_init, "w") as f: f.write(c)

# 2. Inject them safely into YOUR script instead
inference_script = "inference_demo.py"
with open(inference_script, "r") as f: c = f.read()

safe_imports = """
# Safe explicit registrations for Fairseq checkpoints
import fairseq.criterions.masked_lm
import fairseq.criterions.cross_entropy
"""
if "import fairseq.criterions.masked_lm" not in c:
    c = safe_imports + c
    with open(inference_script, "w") as f: f.write(c)

print("✅ Circular imports destroyed! Criterions routed safely to the inference script.")

/content/SAMWISE
✅ Circular imports destroyed! Criterions routed safely to the inference script.


In [27]:
import os
import re

%cd /content/SAMWISE

filepath = "models/samwise.py"

with open(filepath, "r") as f:
    content = f.read()

# 1. Replace explicit .to('mps') with .to('cuda')
content = re.sub(r"\.to\(['\"]mps['\"]\)", ".to('cuda')", content)

# 2. Replace explicit device='mps' with device='cuda'
content = re.sub(r"device\s*=\s*['\"]mps['\"]", "device='cuda'", content)

with open(filepath, "w") as f:
    f.write(content)

print("✅ Hardcoded Apple Silicon ('mps') dependencies neutralized! Tensors routed to CUDA.")

/content/SAMWISE
✅ Hardcoded Apple Silicon ('mps') dependencies neutralized! Tensors routed to CUDA.


In [28]:
%cd /content/SAMWISE

!python inference_demo.py \
    --device "cuda" \
    --input_path "../data/processed_clip.mp4" \
    --text_prompts "बिल्ली पेड़ पर चढ़ रही है"

/content/SAMWISE
2026-05-28 19:24:06.798802: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading explainability model (mlx-community/Qwen2.5-1.5B-Instruct-4bit) into unified memory...
🔄 Intercepted MLX load request for 'mlx-community/Qwen2.5-1.5B-Instruct-4bit'.
🚀 Routing to HuggingFace Transformers on CUDA...
Loading weights: 100% 338/338 [00:01<00:00, 194.73it/s, Materializing param=model.norm.weight]
Model loaded successfully.

Specified path is a video or folder with frames, using video-level configuration
Start inference
Extracting frames from .mp4 in frames_processed_clip with ffmpeg...
frames_processed_clip already exists, using frames in that folder
Begin inference on 107 frames
  7% 1/14 [00:06<01:29,  6.90s/it]
[Frames [8, 9, 10, 11

In [30]:
import os
from moviepy.editor import ImageSequenceClip

%cd /content/SAMWISE

# The exact folder where your PNG frames are sitting
output_dir = "demo_output/बिल्ली_पेड़_पर_चढ़_रही_है"

# Grab only the .png files and sort them numerically
images = [
    os.path.join(output_dir, img)
    for img in sorted(os.listdir(output_dir))
    if img.endswith('.png')
]

if not images:
    print(f"❌ Could not find any .png files in '{output_dir}'.")
else:
    print(f"✅ Found {len(images)} PNG frames! Stitching the video now...")

    # Stitch the frames into a 10fps MP4
    clip = ImageSequenceClip(images, fps=10)
    clip.write_videofile("demo_output/final_segmented_video.mp4", codec="libx264")

    print("🎉 SUCCESS! Video saved to demo_output/final_segmented_video.mp4")

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



/content/SAMWISE
✅ Found 107 PNG frames! Stitching the video now...
Moviepy - Building video demo_output/final_segmented_video.mp4.
Moviepy - Writing video demo_output/final_segmented_video.mp4



Moviepy - Done !
Moviepy - video ready demo_output/final_segmented_video.mp4
🎉 SUCCESS! Video saved to demo_output/final_segmented_video.mp4
